In [25]:
import pandas as pd

# Load RNA-RNA interaction file
file_path = "RNAInter_RNA_RNA.txt"

# Try common delimiters (RNAInter files are usually tab-delimited)
df = pd.read_csv(file_path, sep='\t', low_memory=False)

# Display basic info
print(f"Number of interactions: {len(df)}")
df.head()


Number of interactions: 9484609


,RNAInterID,Interactor1.Symbol,Category1,Species1,Interactor2.Symbol,Category2,Species2,Raw_ID1,Raw_ID2,score,strong,weak,predict
0,RR00000169,(Ca)N,repeats,Mus musculus,B1_Mus1,repeats,Mus musculus,chr4(CA)n125049227,chr15B1_Mus136600585,0.23,NaN,MARIO,NaN
1,RR00000238,(Ca)N,repeats,Mus musculus,B1_Mus1,repeats,Mus musculus,chr8(CA)n67899847,chr8B1_Mus167899573,0.23,NaN,MARIO,NaN
2,RR00000198,(Ca)N,repeats,Mus musculus,B1_Mus1,repeats,Mus musculus,chr6(CA)n135011645,chr6B1_Mus1135012241,0.23,NaN,MARIO,NaN
3,RR00000231,(Ca)N,repeats,Mus musculus,B1_Mus1,repeats,Mus musculus,chr8(CA)n114753246,chr8B1_Mus1114753067,0.23,NaN,MARIO,NaN
4,RR00000249,(Ca)N,repeats,Mus musculus,B3,repeats,Mus musculus,chr9(CA)n21755504,chr9B321755686,0.23,NaN,MARIO,NaN


In [26]:
# Show column names
print("Columns:", df.columns.tolist())

# Inspect data types
df.dtypes


Columns: ['RNAInterID', 'Interactor1.Symbol', 'Category1', 'Species1', 'Interactor2.Symbol', 'Category2', 'Species2', 'Raw_ID1', 'Raw_ID2', 'score', 'strong', 'weak', 'predict']


RNAInterID             object
Interactor1.Symbol     object
Category1              object
Species1               object
Interactor2.Symbol     object
Category2              object
Species2               object
Raw_ID1                object
Raw_ID2                object
score                 float64
strong                 object
weak                   object
predict                object
dtype: object

In [27]:
# Check unique categories in both interactors
cat1_unique = df['Category1'].unique()
cat2_unique = df['Category2'].unique()

# Optionally, combined unique categories across both columns
combined_categories = set(cat1_unique) | set(cat2_unique)
print("All unique RNA categories in dataset:", combined_categories)



All unique RNA categories in dataset: {'unknown', 'TEC', 'snoRNA', 'unassigned RNA', 'piRNA', 'rRNA', 'Category1', 'eRNA', 'sRNA', 'snRNA', 'tRNA', 'repeats', 'ribozyme', 'antisense', 'scaRNA', 'lncRNA', 'miscRNA', 'TR_C_gene', 'scRNA', 'Category2', 'sncRNA', 'ncRNA', 'protein', 'Protein', 'shRNA', 'circRNA', 'mRNA', 'TF', 'tRF', 'pseudo', 'miRNA', 'PCG', 'processed_transcript', 'lincRNA', 'others'}


In [28]:
exclude_categories = {'Category1', 'Category2', 'unknown', 'unassigned RNA', 'others', 'Protein', 'protein',
                      'PCG', # Protein-coding Gene
                      'TF', # transcription factor gene
                      'TR_C_gene', # T-cell receptor constant region gene; immune-related.
                      'antisense', # transcripts complementary to sense strands of annotated genes.
                      'TEC', # “to be experimentally confirmed,” provisional transcripts.
                      'repeats' # transcripts overlapping repetitive genomic elements.
                     }

# Filter out unwanted categories from both interactors
df_filt = df[~df['Category1'].isin(exclude_categories) & ~df['Category2'].isin(exclude_categories)]

print(f"Rows after removing not-clear categories: {len(df_filt)}")


Rows after removing not-clear categories: 9429474


In [29]:
# Check unique categories in both interactors
cat1_unique = df_filt['Category1'].unique()
cat2_unique = df_filt['Category2'].unique()

# Optionally, combined unique categories across both columns
combined_categories = set(cat1_unique) | set(cat2_unique)
print("All unique RNA categories in dataset:", combined_categories)

All unique RNA categories in dataset: {'snoRNA', 'piRNA', 'rRNA', 'eRNA', 'sRNA', 'snRNA', 'tRNA', 'ribozyme', 'scaRNA', 'lncRNA', 'miscRNA', 'scRNA', 'sncRNA', 'ncRNA', 'shRNA', 'circRNA', 'mRNA', 'tRF', 'pseudo', 'miRNA', 'processed_transcript', 'lincRNA'}


In [30]:
# initial count
print(f"RNA–RNA interactions: {len(df_filt)}")

# keep rows where each interactor has at least one identifier
mask_valid_1 = df_filt[['Raw_ID1', 'Interactor1.Symbol']].notna().any(axis=1)
mask_valid_2 = df_filt[['Raw_ID2', 'Interactor2.Symbol']].notna().any(axis=1)

df_filt = df_filt[mask_valid_1 & mask_valid_2]

# final count
print(f"RNA–RNA interactions: {len(df_filt)}")


RNA–RNA interactions: 9429474
RNA–RNA interactions: 9429405


In [31]:
parsed = []

def parse_ids(id_value):
    if pd.isna(id_value):
        return [None]
    elif ':' in id_value:
        prefix, ids_str = id_value.split(':', 1)
        return [f"{prefix}:{id_.strip()}" for id_ in ids_str.split(';')]
    else:
        parsed.append(id_value)
        return [id_value.strip()]

# Add list columns
df_filt['Raw_ID1_list'] = df_filt['Raw_ID1'].apply(parse_ids)
df_filt['Raw_ID2_list'] = df_filt['Raw_ID2'].apply(parse_ids)


In [32]:
# we need to standardise the species to TAXIDs
# 1. See unique species
df_filt.replace({'-': None}, inplace=True)
unique_species = pd.unique(df_filt[['Species1', 'Species2']].values.ravel())
print("Unique species:", len(unique_species))

from ete3 import NCBITaxa
ncbi = NCBITaxa()

unique_species_taxids = {}
missing = []

for test_species in unique_species:
    # Skip placeholders
    if test_species in [None]:
        continue

    # Handle known special cases first
    if test_species == 'Solanum tuberosum L.':
        unique_species_taxids[test_species] = "4113"
        continue
    if test_species == 'Rhodobacter sphaeroides ATCC 17025':
        unique_species_taxids[test_species] = "1061"
        continue

    if test_species == 'Luffa cylindrica':
        unique_species_taxids[test_species] = "3669"
        continue

    # Query NCBI
    try:
        taxid = str(ncbi.get_name_translator([test_species])[test_species][0])
        unique_species_taxids[test_species] = taxid
    except KeyError:
        missing.append(test_species)

print("Mapped species (unique -1 for None):", len(unique_species_taxids))
print("Missing species:", missing)


Unique species: 135
Mapped species (unique -1 for None): 134
Missing species: []


In [33]:
# Assume df is your original dataframe
# unique_species_taxids: dict {species_name -> taxid}

df_filt['Species1_TaxID'] = df_filt['Species1'].map(unique_species_taxids)
df_filt['Species2_TaxID'] = df_filt['Species2'].map(unique_species_taxids)

unmapped_s1 = df_filt.loc[df_filt['Species1'].notna() & df_filt['Species1_TaxID'].isna(), 'Species1'].unique()
unmapped_s2 = df_filt.loc[df_filt['Species2'].notna() & df_filt['Species2_TaxID'].isna(), 'Species2'].unique()

print("Unmapped Species1:", unmapped_s1)
print("Unmapped Species2:", unmapped_s2)

Unmapped Species1: []
Unmapped Species2: []


In [34]:
print(f"RNA–RNA interactions: {len(df_filt)}")
# removing NAN
df_filt = df_filt.dropna(subset=['Species1_TaxID', 'Species2_TaxID'])
print(f"RNA–RNA interactions: {len(df_filt)}")

RNA–RNA interactions: 9429405
RNA–RNA interactions: 9429404


In [35]:
# Remove mRNA interactions at the 
print(f"RNA–RNA interactions: {len(df_filt)}")
df_filt = df_filt[
    (df_filt['Category1'] != 'mRNA') &
    (df_filt['Category2'] != 'mRNA')
].copy()
print(f"RNA–RNA interactions: {len(df_filt)}")

# Remove circRNA interactions at the beginning, 
df_filt = df_filt[
    (df_filt['Category1'] != 'circRNA') &
    (df_filt['Category2'] != 'circRNA')
].copy()
print(f"RNA–RNA interactions: {len(df_filt)}")

# Remove circRNA interactions at the beginning, 
df_filt = df_filt[
    (df_filt['Category1'] != 'tRF') &
    (df_filt['Category2'] != 'tRF')
].copy()
print(f"RNA–RNA interactions: {len(df_filt)}")

RNA–RNA interactions: 9429404
RNA–RNA interactions: 205295
RNA–RNA interactions: 203041
RNA–RNA interactions: 202537


In [36]:
# Check unique categories in both interactors
cat1_unique = df_filt['Category1'].unique()
cat2_unique = df_filt['Category2'].unique()

# Optionally, combined unique categories across both columns
combined_categories = set(cat1_unique) | set(cat2_unique)
print("All unique RNA categories in dataset:", combined_categories)

RNA_TYPE_MAP = {
    'processed_transcript': 'ncRNA',
    'miscRNA': 'ncRNA',
}

# Apply standardisation
df_filt['Category1_std'] = df_filt['Category1'].replace(RNA_TYPE_MAP)
df_filt['Category2_std'] = df_filt['Category2'].replace(RNA_TYPE_MAP)

# Recompute unique categories after mapping
cat1_std = set(df_filt['Category1_std'].unique())
cat2_std = set(df_filt['Category2_std'].unique())

print("Standardised categories:", cat1_std | cat2_std)

assert not df_filt[['Category1_std', 'Category2_std']].isna().any().any()

All unique RNA categories in dataset: {'processed_transcript', 'scaRNA', 'lncRNA', 'snoRNA', 'miscRNA', 'scRNA', 'sncRNA', 'ncRNA', 'piRNA', 'rRNA', 'pseudo', 'miRNA', 'sRNA', 'snRNA', 'tRNA', 'lincRNA', 'ribozyme'}
Standardised categories: {'scaRNA', 'lncRNA', 'snoRNA', 'scRNA', 'sncRNA', 'ncRNA', 'piRNA', 'rRNA', 'pseudo', 'miRNA', 'sRNA', 'snRNA', 'tRNA', 'lincRNA', 'ribozyme'}


In [37]:
# DATASET IS READY.

# Now we need to extract sequences, we start by taking unique of ('Interactor1.Symbol', 'Category1', 'Species1_TaxID', 'Raw_ID1_list')
# We do not care about 1 and 2, we can create the final database later.

def normalize_id_list(x):
    if isinstance(x, list):
        return tuple(sorted(str(i) for i in x if i is not None))
    return tuple()
df_filtbackup = df_filt

df_filt = df_filt.copy()
df_filt['Raw_ID1_key'] = df_filt['Raw_ID1_list'].apply(normalize_id_list)
df_filt['Raw_ID2_key'] = df_filt['Raw_ID2_list'].apply(normalize_id_list)

rna_entities_1 = (
    df_filt.loc[:, ['Interactor1.Symbol', 'Category1', 'Species1_TaxID', 'Raw_ID1_key']]
    .drop_duplicates()
    .rename(columns={
        'Interactor1.Symbol': 'Symbol',
        'Category1': 'Category',
        'Species1_TaxID': 'Species',
        'Raw_ID1_key': 'Raw_ID_key'
    })
)

rna_entities_2 = (
    df_filt.loc[:, ['Interactor2.Symbol', 'Category2', 'Species2_TaxID', 'Raw_ID2_key']]
    .drop_duplicates()
    .rename(columns={
        'Interactor2.Symbol': 'Symbol',
        'Category2': 'Category',
        'Species2_TaxID': 'Species',
        'Raw_ID2_key': 'Raw_ID_key'
    })
)

rna_entities = (
    pd.concat([rna_entities_1, rna_entities_2], ignore_index=True)
    .drop_duplicates()
)

print(f"Unique RNA entities: {len(rna_entities)}")

rna_entities['RNA_IDs'] = rna_entities['Raw_ID_key'].apply(list)

def classify_ids(id_list):
    return {
        'miRBase': [],
        'NCBI': [],
        'Ensembl': [],
        'circBase': [],
        'tRFdb': [],
        'GenBank': [],
        'Other': [],
        'RNAcentral': []   # ← to be filled after resolution
    }

def populate_buckets(id_list):
    buckets = classify_ids(id_list)
    for rid in id_list:
        if rid.startswith('miRBase:'):
            buckets['miRBase'].append(rid.split(':', 1)[1])
        elif rid.startswith('NCBI:'):
            buckets['NCBI'].append(rid.split(':', 1)[1])
        elif rid.startswith('Ensembl:'):
            buckets['Ensembl'].append(rid.split(':', 1)[1])
        elif rid.startswith('circBase:'):
            buckets['circBase'].append(rid.split(':', 1)[1])
        elif rid.startswith('tRFdb:'):
            buckets['tRFdb'].append(rid.split(':', 1)[1])
        elif rid.startswith('GenBank:'):
            buckets['GenBank'].append(rid.split(':', 1)[1])
        else:
            buckets['Other'].append(rid)
    return buckets

rna_entities['ID_buckets'] = rna_entities['RNA_IDs'].apply(populate_buckets)

import pandas as pd
df_filt.to_pickle("interactions_processed_noRNAcentralIDs.pkl")

Unique RNA entities: 35450


In [38]:
from collections import defaultdict

# Initialize counter
db_counts = defaultdict(int)

for buckets in rna_entities['ID_buckets']:
    for db, ids in buckets.items():
        if db != 'RNAcentral':  # exclude placeholder
            db_counts[db] += len(set(ids))  # count unique IDs per entity

# Print summary
for db, count in db_counts.items():
    print(f"{db}: {count} unique IDs across entities")


miRBase: 6731 unique IDs across entities
NCBI: 21192 unique IDs across entities
Ensembl: 3889 unique IDs across entities
circBase: 1 unique IDs across entities
tRFdb: 0 unique IDs across entities
GenBank: 3 unique IDs across entities
Other: 0 unique IDs across entities


In [39]:
from collections import defaultdict

# Initialize counter
db_counts = defaultdict(int)

for buckets in rna_entities['ID_buckets']:
    for db, ids in buckets.items():
        if db != 'RNAcentral' and ids:  # exclude placeholder and empty lists
            db_counts[db] += 1  # count the row/entity, not unique IDs

# Print summary
for db, count in db_counts.items():
    print(f"{db}: {count} rows containing IDs")


Ensembl: 3889 rows containing IDs
NCBI: 21192 rows containing IDs
miRBase: 6723 rows containing IDs
GenBank: 3 rows containing IDs
circBase: 1 rows containing IDs


In [40]:
'''
To-discuss: Assumption: 
all annotated isoforms (transcripts) of a gene are considered potential interactors 
when RNAInter reports a gene-level interaction.

We could think of some filters on the isoforms. For now we download them all, and keep track of isoforms-per-interaction (A list of sequences)
'''

'\nTo-discuss: Assumption: \nall annotated isoforms (transcripts) of a gene are considered potential interactors \nwhen RNAInter reports a gene-level interaction.\n\nWe could think of some filters on the isoforms. For now we download them all, and keep track of isoforms-per-interaction (A list of sequences)\n'

<h3>Ensembl IDs to Ensemble transcripts to RNAcentral IDs</h3>

In [41]:
# Construct DUCK-DB based on RNAcentral BEFORE this step!!
import sys
sys.path.append("rnacentral-etl")

from rnacentral import QueryRnaCentral

qrc = QueryRnaCentral(base_dir="rnacentral_data/current_release", db_path="rnacentral-etl/rnacentral.duckdb")

In [42]:
'''
Adapt the code below based on the instructions:
Extract all RNAcentral IDS of the ensemble transcripts related to ensemble GeneIDs that satisfy the conditions (Tax ID and rna-type)
Extract from row.get("ID_buckets", {}).get("Ensembl", [])
and place in RNAcentral a list for each transcript, so that we have a list of lists. row.get("ID_buckets", {}).get("RNAcentral", [])

We are not sure if all ENSEMBL IDs we have are gene ids, so first try with gene, if nothing try with transcript ID.
'''

# Usefoul code: 
taxid= "10090"
res = qrc.ensembl_gene_to_rnacentral_transcripts("ENSMUSG00000065862", taxid) #for ENS. gene ID
# qrc.ensembl_to_rnacentral('ENSMUST00000083928', taxid) # ENSMUSG00000065862 for ENS transcript ID
'''
Ensure 'rna_type' returned by query rna central is the same rna_type of the row.
First look if the rnacentral type is in the standard categories: Standardised categories: {'tRNA', 'miRNA', 'rRNA', 'pseudo', 'lncRNA', 'piRNA', 'tRF', 'mRNA', 'sRNA', 'lincRNA', 'circRNA', 'ncRNA', 'ribozyme', 'snoRNA', 'scaRNA', 'snRNA', 'scRNA', 'sncRNA'}
If not print it because we need to standardise everything.
'''

res

,RNAcentral_ID,rna_type,taxid,external_db
0,URS0000639553,snoRNA,10090,ENSEMBL


In [43]:
STANDARD_RNA_TYPES = {
    "tRNA", "miRNA", "rRNA", "pseudo", "lncRNA", "piRNA",
    "sRNA", "lincRNA", "ncRNA", "ribozyme", "snoRNA",
    "scaRNA", "snRNA", "scRNA", "sncRNA"
}
ACCEPTABLE_TYPE_MAP = {
    "SRP_RNA": ["ncRNA"],
    "RNase_MRP_RNA": ["ncRNA", "ribozyme"],
    "pre_miRNA": ["miRNA"],          # precursor -> mature
    "scaRNA": ["snoRNA", "ncRNA"],
    "snRNA": ["ncRNA"]               # small nuclear RNA -> generic ncRNA
}


def get_rnacentral_ids(row):
    """
    Populate row['ID_buckets']['RNAcentral'] with:
    a list of lists (one list per Ensembl ID).
    """
    row = row.copy()
    id_buckets = row.get("ID_buckets", {})
    ensembl_ids = id_buckets.get("Ensembl", [])
    taxid = str(row.get("Species"))
    row_rna_type = row.get("Category")

    if not ensembl_ids or taxid is None:
        return row

    rnacentral_per_ensembl = []
    
    for ensembl_id in ensembl_ids:
        rnacentral_ids = set()

        try:
            # 1) try Ensembl gene → RNAcentral transcripts
            df = qrc.ensembl_gene_to_rnacentral_transcripts(
                ensembl_id, taxid
            )

            # 2) fallback: Ensembl transcript → RNAcentral
            if df is None or df.empty:
                df = qrc.ensembl_to_rnacentral(
                    ensembl_id, taxid
                )
                if df is not None and not df.empty:
                    print('Transcript detected')
                    print(df)

            if df is None or df.empty:
                continue

            for _, r in df.iterrows():
                rc_id = r.get("RNAcentral_ID")
                rc_type = r.get("rna_type")

                if pd.isna(rc_id) or pd.isna(rc_type):
                    print('rnacentral data not valid')
                    continue

                if rc_type not in STANDARD_RNA_TYPES:
                    print(f"[UNSTANDARDISED RNA TYPE] {rc_type}, row type in this case: {row_rna_type}")
                
                # Check if the rc_type is OK for the row type
                if rc_type == row_rna_type:
                    rnacentral_ids.add(rc_id)
                elif rc_type in ACCEPTABLE_TYPE_MAP and row_rna_type in ACCEPTABLE_TYPE_MAP[rc_type]:
                    # defensible bypass
                    rnacentral_ids.add(rc_id)
                else:
                    # truly mismatched type — log for review
                    print(f"[NO TYPE MATCH] {rc_type}, row type in this case: {row_rna_type}")

        except Exception:
            continue

        if rnacentral_ids:
            rnacentral_per_ensembl.append(list(rnacentral_ids))

    id_buckets["RNAcentral"] = rnacentral_per_ensembl if rnacentral_per_ensembl else []
    row["ID_buckets"] = id_buckets

    return row


rna_entities_trial = rna_entities.apply(
    get_rnacentral_ids, axis=1
)


[NO TYPE MATCH] lncRNA, row type in this case: pseudo
[NO TYPE MATCH] lncRNA, row type in this case: pseudo
[NO TYPE MATCH] ncRNA, row type in this case: snRNA
[NO TYPE MATCH] ncRNA, row type in this case: snRNA
[NO TYPE MATCH] sRNA, row type in this case: miRNA
[NO TYPE MATCH] ncRNA, row type in this case: snRNA
[UNSTANDARDISED RNA TYPE] misc_RNA, row type in this case: snRNA
[NO TYPE MATCH] misc_RNA, row type in this case: snRNA


In [78]:
from collections import defaultdict
import numpy as np

def compute_stats(df, target_db="RNAcentral"):
    """
    Compute row-level and ID-level statistics for ID_buckets.
    Row = entity (primary unit).
    """

    # -------------------------
    # Row-level statistics
    # -------------------------
    db_rows_with_ids = defaultdict(int)
    db_ids_per_row = defaultdict(list)

    for _, row in df.iterrows():
        buckets = row.get("ID_buckets", {})
        for db, ids in buckets.items():
            if not ids:
                continue

            # count this row once per DB
            db_rows_with_ids[db] += 1

            # number of IDs in this row for this DB
            n_ids = 0
            for x in ids:
                n_ids += len(x) if isinstance(x, list) else 1

            db_ids_per_row[db].append(n_ids)

    print("=== Row-level (entity-level) statistics ===")
    for db, counts in db_ids_per_row.items():
        arr = np.array(counts)
        print(
            f"{db}: rows={db_rows_with_ids[db]} | "
            f"IDs/row: max={arr.max()}, mean={arr.mean():.2f}, median={np.median(arr)}"
        )

    # -------------------------
    # Row-level bins (IDs per row)
    # -------------------------
    print("\n=== IDs per row (binned) ===")
    for db, counts in db_ids_per_row.items():
        count_1 = np.sum(np.array(counts) == 1)
        count_2 = np.sum(np.array(counts) == 2)
        count_3_10 = np.sum((np.array(counts) >= 3) & (np.array(counts) <= 10))
        count_gt10 = np.sum(np.array(counts) > 10)

        print(
            f"{db}: 1={count_1}, 2={count_2}, "
            f"3–10={count_3_10}, >10={count_gt10}"
        )

    # -------------------------
    # ID-level statistics (secondary)
    # -------------------------
    db_unique_ids = defaultdict(set)

    for _, row in df.iterrows():
        buckets = row.get("ID_buckets", {})
        for db, ids in buckets.items():
            for x in ids:
                if isinstance(x, list):
                    db_unique_ids[db].update(x)
                else:
                    db_unique_ids[db].add(x)

    print("\n=== ID-level (secondary) statistics ===")
    for db, ids in db_unique_ids.items():
        print(f"{db}: {len(ids)} unique IDs")

    # -------------------------
    # Sequence availability (target DB)
    # -------------------------
    if target_db:
        has_seq = sum(
            bool(row.get("ID_buckets", {}).get(target_db))
            for _, row in df.iterrows()
        )
        no_seq = len(df) - has_seq

        print("\n=== Sequence availability ===")
        print(f"{target_db}: rows with IDs = {has_seq}")
        print(f"{target_db}: rows without IDs = {no_seq}")
        print(f"{target_db}: fraction retained = {has_seq / len(df):.3f}")

    # -------------------------
    # Sequence availability (target DB OR fallback sequence)
    # -------------------------
    print('new sequence avibility counting sequences too')
    if target_db:
        has_retained = 0
    
        for _, row in df.iterrows():
            id_buckets = row.get("ID_buckets", {})
            has_ids = bool(id_buckets.get(target_db))
    
            seqs = row.get("sequences")
            has_sequences = isinstance(seqs, list) and len(seqs) >= 1
    
            if has_ids or has_sequences:
                has_retained += 1
    
        not_retained = len(df) - has_retained
    
        print("\n=== Sequence availability ===")
        print(f"{target_db} IDs OR fallback sequence present = {has_retained}")
        print(f"Rows without sequence information = {not_retained}")
        print(f"Fraction retained = {has_retained / len(df):.3f}")
    


In [52]:
compute_stats(rna_entities_trial)

=== Row-level (entity-level) statistics ===
Ensembl: rows=3889 | IDs/row: max=1, mean=1.00, median=1.0
RNAcentral: rows=2137 | IDs/row: max=249, mean=4.88, median=1.0
NCBI: rows=21192 | IDs/row: max=1, mean=1.00, median=1.0
miRBase: rows=6723 | IDs/row: max=3, mean=1.00, median=1.0
GenBank: rows=3 | IDs/row: max=1, mean=1.00, median=1.0
circBase: rows=1 | IDs/row: max=1, mean=1.00, median=1.0

=== IDs per row (binned) ===
Ensembl: 1=3889, 2=0, 3–10=0, >10=0
RNAcentral: 1=1382, 2=183, 3–10=352, >10=220
NCBI: 1=21192, 2=0, 3–10=0, >10=0
miRBase: 1=6716, 2=6, 3–10=1, >10=0
GenBank: 1=3, 2=0, 3–10=0, >10=0
circBase: 1=1, 2=0, 3–10=0, >10=0

=== ID-level (secondary) statistics ===
Ensembl: 3889 unique IDs
RNAcentral: 10381 unique IDs
NCBI: 20784 unique IDs
miRBase: 6584 unique IDs
GenBank: 3 unique IDs
circBase: 1 unique IDs

=== Sequence availability ===
RNAcentral: rows with IDs = 2137
RNAcentral: rows without IDs = 33313
RNAcentral: fraction retained = 0.060


In [46]:
rna_entities_trial.to_pickle("rna_entities_trial.pkl")

In [74]:
# Construct DUCK-DB based on RNAcentral BEFORE this step!!
import sys
import pandas as pd
sys.path.append("rnacentral-etl")

from rnacentral import QueryRnaCentral

qrc = QueryRnaCentral(base_dir="rnacentral_data/current_release", db_path="rnacentral-etl/rnacentral.duckdb")

rna_entities_trial = pd.read_pickle("rna_entities_trial.pkl")

In [48]:
# -----------------------------
# Load NCBI → Ensembl mapping
# -----------------------------
ncbi_to_ensemble = pd.read_csv(
    "gene2ensembl",
    sep="\t",
    dtype=str,
    comment="#",
    header=None,
    low_memory=False,
)

ncbi_to_ensemble.columns = [
    "taxid",
    "geneid",
    "ensembl_gene",
    "refseq_rna_version",
    "ensembl_transcript_id",
    "refseq_protein_version",
    "ensembl_protein_id",
]

# enforce string typing
ncbi_to_ensemble["taxid"] = ncbi_to_ensemble["taxid"].astype(str)
ncbi_to_ensemble["geneid"] = ncbi_to_ensemble["geneid"].astype(str)

# collect all GeneIDs used in your dataset
all_ncbi_ids = set()

for buckets in rna_entities_trial["ID_buckets"]:
    all_ncbi_ids.update(map(str, buckets.get("NCBI", [])))

print(f"{len(all_ncbi_ids)} unique NCBI GeneIDs")

ncbi_to_ensemble_small = ncbi_to_ensemble[
    (ncbi_to_ensemble["ensembl_gene"] != "-") &
    (ncbi_to_ensemble["geneid"].isin(all_ncbi_ids))
]

ncbi2ens_gene = (
    ncbi_to_ensemble_small
    .groupby(["taxid", "geneid"])["ensembl_gene"]
    .unique()
    .to_dict()
)
print(len(ncbi2ens_gene))

import pickle
from functools import lru_cache

# key = rc_type from RNAcentral, value = list of row types it can match
ACCEPTABLE_TYPE_MAP = {
    "SRP_RNA": ["ncRNA"],
    "RNase_MRP_RNA": ["ncRNA", "ribozyme"],
    "pre_miRNA": ["miRNA"],          # precursor -> mature
    "scaRNA": ["snoRNA", "ncRNA"],
    "snRNA": ["ncRNA"]               # small nuclear RNA -> generic ncRNA
}

20784 unique NCBI GeneIDs
13698


<h3>NCBI IDs to refSeq IDs to RNAcentral IDs</h3>

In [49]:
from collections import defaultdict
from functools import lru_cache

gene2refseq_index = defaultdict(list)

with open("gene2refseq", "r") as f:   # <-- FIX: no gzip
    for line in f:
        if line.startswith("#"):
            continue
        cols = line.rstrip("\n").split("\t")

        taxid = cols[0]
        geneid = cols[1]
        rna_acc = cols[3]   # RNA_nucleotide_accession
        status = cols[2]

        if status == "SUPPRESSED":
            continue

        if rna_acc == "-":
            continue

        if rna_acc.startswith(("NR_", "XR_")):
            rna_type = "ncRNA"
        #elif rna_acc.startswith(("NM_", "XM_")):
        #    continue
            #rna_type = "mRNA"
        else:
            continue

        gene2refseq_index[(taxid, geneid)].append({
            "refseq_id": rna_acc,
            "rna_type": rna_type,
            "status": status
        })



In [75]:
@lru_cache(maxsize=100_000)
def ncbi_gene_to_refseq_transcripts(ncbi_gene_id, taxid):
    key = (str(ncbi_gene_id), str(taxid))  # cast to str
    return gene2refseq_index.get(key, [])

print(ncbi_gene_to_refseq_transcripts(2711, 102577949))

refseq_id = "NR_036832"  # example RefSeq
taxid = "58123"
print(f"\nRefSeq to RNAcentral (RefSeq={refseq_id}, taxid={taxid}):")
df_ref = qrc.refseq_to_rnacentral(refseq_id, taxid)
print(df_ref)

[{'refseq_id': 'XR_003065960.2', 'rna_type': 'ncRNA', 'status': 'MODEL'}, {'refseq_id': 'XR_371036.4', 'rna_type': 'ncRNA', 'status': 'MODEL'}]

RefSeq to RNAcentral (RefSeq=NR_036832, taxid=58123):
   RNAcentral_ID rna_type
0  URS0000000036     rRNA


In [76]:
def extract_refseq_transcript_type(refseq_id: str):
    try:
        handle = Entrez.efetch(
            db="nuccore",
            id=refseq_id,
            rettype="gb",
            retmode="text"
        )
        record = SeqIO.read(handle, "genbank")
        handle.close()

        for feature in record.features:
            if feature.type == "ncRNA":
                qualifiers = feature.qualifiers
                
                if "ncRNA_class" in qualifiers:
                    return qualifiers["ncRNA_class"][0]

                if "product" in qualifiers:
                    return qualifiers["product"][0]

        return None

    except Exception as e:
        print("Transcript fetch failed:", refseq_id, e)
        return None


In [ ]:
from Bio import Entrez, SeqIO
from io import StringIO

Entrez.email = "marco.nicolini@unimi.it" 

def fetch_refseq_sequence(refseq_id: str) -> str:
    handle = Entrez.efetch(
        db="nuccore",
        id=refseq_id,
        rettype="fasta",
        retmode="text"
    )
    fasta_text = handle.read()
    handle.close()
    
    record = SeqIO.read(StringIO(fasta_text), "fasta")
    return str(record.seq)
    
def normalize_sequence(seq: str) -> str:
    seq = seq.upper()
    return seq

errors_standard_types = []
errors_no_match_types = []

def get_rnacentral_from_ncbi_ids_refseq(row):
    row = row.copy()
    id_buckets = row.get("ID_buckets", {})
    ncbi_ids = id_buckets.get("NCBI", [])
    taxid = str(row.get("Species"))
    row_rna_type = row.get("Category")

    if not ncbi_ids or not taxid:
        return row

    # do not overwrite existing RNAcentral mappings
    if id_buckets.get("RNAcentral"):
        return row

    rnacentral_per_ncbi = []
    sequences = []

    for ncbi_id in ncbi_ids:
        rnacentral_ids = set()
        try:
            refseq_records = ncbi_gene_to_refseq_transcripts(
                taxid, ncbi_id
            )
            if not refseq_records:
                continue

            for rec in refseq_records:
                refseq_id = rec.get("refseq_id")
                refseq_id = refseq_id.split(".")[0]
                status = rec.get("status")
                

                if not refseq_id:
                    continue

                df_rnacentral = qrc.refseq_to_rnacentral(str(refseq_id), str(taxid))

                if df_rnacentral is None or df_rnacentral.empty:
                    if not rnacentral_ids: # if we have good RNAcentral matches we can skip
                        try:
                            # download sequence
                            seq = fetch_refseq_sequence(str(refseq_id))
                            seq = normalize_sequence(seq)
                            seq_Us = seq.replace("T", "U")
                            
                        except Exception as e:
                            print("Sequence fetch failed:", refseq_id, e)
                            continue
                        
                        df_rnacentral = qrc.sequence_to_rnacentral(seq, str(taxid))
                        
                        if df_rnacentral is None or df_rnacentral.empty:
                            df_rnacentral = qrc.sequence_to_rnacentral(seq_Us, str(taxid))
                        if df_rnacentral is None or df_rnacentral.empty:
                            # we check for REFSEQ TYPE, no RNAcentral Mapping
                            refseq_type = extract_refseq_transcript_type(str(refseq_id))
                            if refseq_type and refseq_type.lower() == row_rna_type.lower():
                                sequences.append(seq_Us)
                            continue 
                        
                # For loop for Rnacentral items:
                for _, row_rc in df_rnacentral.iterrows():
                    rnacentral_id = row_rc.get('RNAcentral_ID')
                    rc_type = row_rc.get('rna_type')

                    if rc_type not in STANDARD_RNA_TYPES:
                        errors_standard_types.append(
                            f"[UNSTANDARDISED RNA TYPE] {rc_type} (refseq_id:{refseq_id})"
                        )
                        
                    # Check if the rc_type is OK for the row type
                    if rc_type == row_rna_type:
                        rnacentral_ids.add(rnacentral_id)
                    elif rc_type in ACCEPTABLE_TYPE_MAP and row_rna_type in ACCEPTABLE_TYPE_MAP[rc_type]:
                        # defensible bypass
                        rnacentral_ids.add(rnacentral_id)
                    else:
                        # truly mismatched type — log for review
                        errors_no_match_types.append(f"[no match RNA TYPE] {rc_type} for row type {row_rna_type}")

        except Exception as e:
            print("Error:", e)
            continue

        if rnacentral_ids:
            rnacentral_per_ncbi.append(list(rnacentral_ids))

    id_buckets["RNAcentral"] = rnacentral_per_ncbi
    row["ID_buckets"] = id_buckets
    if not rnacentral_per_ncbi and sequences:
        row["sequences"] = sequences
    return row

from tqdm.auto import tqdm
tqdm.pandas()

rna_entities_trial_stepTwo = rna_entities_trial.progress_apply(
    get_rnacentral_from_ncbi_ids_refseq,
    axis=1
)

rna_entities_trial_stepTwo.to_pickle("rna_entities_trial_stepTwo.pkl")

compute_stats(rna_entities_trial_stepTwo)


In [79]:
compute_stats(rna_entities_trial_stepTwo)

=== Row-level (entity-level) statistics ===
Ensembl: rows=3889 | IDs/row: max=1, mean=1.00, median=1.0
RNAcentral: rows=7929 | IDs/row: max=249, mean=2.41, median=1.0
NCBI: rows=21192 | IDs/row: max=1, mean=1.00, median=1.0
miRBase: rows=6723 | IDs/row: max=3, mean=1.00, median=1.0
GenBank: rows=3 | IDs/row: max=1, mean=1.00, median=1.0
circBase: rows=1 | IDs/row: max=1, mean=1.00, median=1.0

=== IDs per row (binned) ===
Ensembl: 1=3889, 2=0, 3–10=0, >10=0
RNAcentral: 1=6027, 2=791, 3–10=856, >10=255
NCBI: 1=21192, 2=0, 3–10=0, >10=0
miRBase: 1=6716, 2=6, 3–10=1, >10=0
GenBank: 1=3, 2=0, 3–10=0, >10=0
circBase: 1=1, 2=0, 3–10=0, >10=0

=== ID-level (secondary) statistics ===
Ensembl: 3889 unique IDs
RNAcentral: 18833 unique IDs
NCBI: 20784 unique IDs
miRBase: 6584 unique IDs
GenBank: 3 unique IDs
circBase: 1 unique IDs

=== Sequence availability ===
RNAcentral: rows with IDs = 7929
RNAcentral: rows without IDs = 27521
RNAcentral: fraction retained = 0.224
new sequence avibility counti

<h3>NCBI IDs to Ensembl IDs to Ensemble transcripts to RNAcentral IDs</h3>

In [80]:
@lru_cache(maxsize=100_000)
def cached_ensembl_gene_to_rnacentral(ensembl_gene, taxid):
    return qrc.ensembl_gene_to_rnacentral_transcripts(ensembl_gene, taxid)

# -----------------------------
# Main function
# -----------------------------
errors_standard_types = []
errors_no_match_types = []
def get_rnacentral_from_ncbi_ids(row):
    """
    Starting from NCBI GeneIDs:
    - map to Ensembl transcripts via gene2ensembl
    - query RNAcentral at transcript level
    - enforce RNA type consistency
    - store list-of-lists in ID_buckets['RNAcentral']
      (one list per NCBI GeneID)
    - append to existing RNAcentral IDs if present
    """
    row = row.copy()   # critical
    id_buckets = row.get("ID_buckets", {})
    ncbi_ids = id_buckets.get("NCBI", [])
    taxid = str(row.get("Species"))
    row_rna_type = row.get("Category")  

    if not ncbi_ids or taxid is None:
        return row

    # skip existing RNAcentral entries, if any
    existing = id_buckets.get("RNAcentral", [])
    if existing:
        #print(existing)
        #print(id_buckets)
        return row
    rnacentral_per_ncbi = []

    for ncbi_id in ncbi_ids:
        rnacentral_ids = set()

        try:
            # NCBI GeneID → Ensembl Gene IDs
            ens_genes = ncbi2ens_gene.get((taxid, str(ncbi_id)), [])
            
            for ensembl_gene in ens_genes:
                #df = qrc.ensembl_gene_to_rnacentral_transcripts(
                #    ensembl_gene, taxid
                #)
                df = cached_ensembl_gene_to_rnacentral(ensembl_gene, taxid)

                if df is None or df.empty:
                    continue

                for _, r in df.iterrows():
                    rc_id = r.get("RNAcentral_ID") or r.get("rnacentral_id")
                    rc_type = r.get("rna_type")
                    
                    if pd.isna(rc_id) or pd.isna(rc_type):
                        continue

                    # Normalize unstandardised RNA types if needed
                    if rc_type not in STANDARD_RNA_TYPES:
                        errors_standard_types.append(f"[UNSTANDARDISED RNA TYPE] {rc_type} for row type {row_rna_type}")
                    
                    # Check if the rc_type is OK for the row type
                    if rc_type == row_rna_type:
                        rnacentral_ids.add(rc_id)
                    elif rc_type in ACCEPTABLE_TYPE_MAP and row_rna_type in ACCEPTABLE_TYPE_MAP[rc_type]:
                        # defensible bypass
                        rnacentral_ids.add(rc_id)
                    else:
                        # truly mismatched type — log for review
                        errors_no_match_types.append(f"[no match RNA TYPE] {rc_type} for row type {row_rna_type}")
        except Exception:
            continue

        # Only append non-empty lists
        if rnacentral_ids:
            rnacentral_per_ncbi.append(list(rnacentral_ids))

    # If nothing found and no previous entries, make it empty
    id_buckets["RNAcentral"] = rnacentral_per_ncbi if rnacentral_per_ncbi else []
    row["ID_buckets"] = id_buckets

    return row

# -----------------------------
# Apply to dataframe
# -----------------------------

from tqdm import tqdm

tqdm.pandas(desc="Querying RNAcentral from NCBI IDs")

rna_entities_trial_stepThree = rna_entities_trial_stepTwo.progress_apply(
    get_rnacentral_from_ncbi_ids,
    axis=1
)

rna_entities_trial_stepThree.to_pickle("rna_entities_trial_stepThree.pkl")

compute_stats(rna_entities_trial_stepThree)

Querying RNAcentral from NCBI IDs: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 35450/35450 [04:37<00:00, 127.66it/s]


=== Row-level (entity-level) statistics ===
Ensembl: rows=3889 | IDs/row: max=1, mean=1.00, median=1.0
RNAcentral: rows=8113 | IDs/row: max=249, mean=2.70, median=1.0
NCBI: rows=21192 | IDs/row: max=1, mean=1.00, median=1.0
miRBase: rows=6723 | IDs/row: max=3, mean=1.00, median=1.0
GenBank: rows=3 | IDs/row: max=1, mean=1.00, median=1.0
circBase: rows=1 | IDs/row: max=1, mean=1.00, median=1.0

=== IDs per row (binned) ===
Ensembl: 1=3889, 2=0, 3–10=0, >10=0
RNAcentral: 1=6084, 2=806, 3–10=910, >10=313
NCBI: 1=21192, 2=0, 3–10=0, >10=0
miRBase: 1=6716, 2=6, 3–10=1, >10=0
GenBank: 1=3, 2=0, 3–10=0, >10=0
circBase: 1=1, 2=0, 3–10=0, >10=0

=== ID-level (secondary) statistics ===
Ensembl: 3889 unique IDs
RNAcentral: 21312 unique IDs
NCBI: 20784 unique IDs
miRBase: 6584 unique IDs
GenBank: 3 unique IDs
circBase: 1 unique IDs

=== Sequence availability ===
RNAcentral: rows with IDs = 8113
RNAcentral: rows without IDs = 27337
RNAcentral: fraction retained = 0.229
new sequence avibility counti

<h3>miRbase IDs to mature miRbase IDs to RNAcentral IDs</h3>

In [ ]:
#import pandas as pd
#rna_entities_trial_stepThree = pd.read_pickle("rna_entities_trial_stepThree.pkl")

In [81]:
qrc.mirbase_to_rnacentral('MIMAT0000434', '9606')

,RNAcentral_ID,rna_type,taxid
0,URS00002620A7,miRNA,9606


In [82]:
from Bio import SeqIO
from collections import defaultdict

# precursor_id -> list of mature IDs (MIMAT…)
precursor_to_mature_ids = defaultdict(list)

# mature_id -> sequence
mature_to_sequence = defaultdict(list)

# precursor_id -> fallback sequences
precursor_to_sequence = defaultdict(list)

for record in SeqIO.parse("miRNA.dat", "embl"):
    precursor_id = record.id  # MI000...

    for feature in record.features:
        if feature.type not in ("miRNA", "mature_miRNA"):
            continue

        seq = feature.location.extract(record).seq.upper().replace("T", "U")
        quals = feature.qualifiers

        mat_acc = quals.get("accession", [None])[0]

        if mat_acc:  # mature miRNA
            mature_to_sequence[mat_acc].append(str(seq))
            precursor_to_mature_ids[precursor_id].append(mat_acc)
        else:
            # precursor-only sequence (fallback)
            precursor_to_sequence[precursor_id].append(str(seq))


In [86]:
def get_mirna_sequences(row):
    row = row.copy()
    id_buckets = row.get("ID_buckets", {})
    taxid = str(row.get("Species"))

    mirbase_ids = id_buckets.get("miRBase", [])
    rnacentral_per_mirbase = []
    seqs = set()

    if not mirbase_ids:
        return row

    rnacentral_ids = set()

    for mid in mirbase_ids:
        mature_ids = []

        if mid.startswith("MIMAT"):
            mature_ids = [mid]
        else:
            # precursor → mature IDs
            mature_ids = precursor_to_mature_ids.get(mid, [])

            # fallback: no mature annotated
            if not mature_ids:
                seqs.update(precursor_to_sequence.get(mid, []))
                continue

        for mature_id in mature_ids:
            # sequences
            seqs.update(mature_to_sequence.get(mature_id, []))

            # RNAcentral mapping
            df_rc = qrc.mirbase_to_rnacentral(mature_id, taxid)
            if df_rc is not None and not df_rc.empty:
                rnacentral_ids.update(df_rc["RNAcentral_ID"].tolist())

    if rnacentral_ids:
        rnacentral_per_mirbase.append(list(rnacentral_ids))

    row["sequences"] = list(seqs)
    id_buckets["RNAcentral"] = rnacentral_per_mirbase
    row["ID_buckets"] = id_buckets

    return row


In [87]:
rna_entities_trial_stepFour = rna_entities_trial_stepThree.apply(
    get_mirna_sequences, axis=1
)

In [88]:
compute_stats(rna_entities_trial_stepFour) # new

=== Row-level (entity-level) statistics ===
Ensembl: rows=3889 | IDs/row: max=1, mean=1.00, median=1.0
RNAcentral: rows=14754 | IDs/row: max=249, mean=1.99, median=1.0
NCBI: rows=21192 | IDs/row: max=1, mean=1.00, median=1.0
miRBase: rows=6723 | IDs/row: max=3, mean=1.00, median=1.0
GenBank: rows=3 | IDs/row: max=1, mean=1.00, median=1.0
circBase: rows=1 | IDs/row: max=1, mean=1.00, median=1.0

=== IDs per row (binned) ===
Ensembl: 1=3889, 2=0, 3–10=0, >10=0
RNAcentral: 1=11931, 2=1595, 3–10=915, >10=313
NCBI: 1=21192, 2=0, 3–10=0, >10=0
miRBase: 1=6716, 2=6, 3–10=1, >10=0
GenBank: 1=3, 2=0, 3–10=0, >10=0
circBase: 1=1, 2=0, 3–10=0, >10=0

=== ID-level (secondary) statistics ===
Ensembl: 3889 unique IDs
RNAcentral: 26075 unique IDs
NCBI: 20784 unique IDs
miRBase: 6584 unique IDs
GenBank: 3 unique IDs
circBase: 1 unique IDs

=== Sequence availability ===
RNAcentral: rows with IDs = 14754
RNAcentral: rows without IDs = 20696
RNAcentral: fraction retained = 0.416
new sequence avibility co

In [89]:
rna_entities_trial_stepFour.to_pickle("rna_entities_trial_stepFour.pkl")

<h3>Final step for sequence conversion: Symbol from RNAInter</h3>

In [90]:
import pandas as pd
#rna_entities_trial_stepFour = pd.read_pickle("rna_entities_trial_stepFour.pkl")

# we start from rna_entities_trial_stepFour
# 1. for each row we search if RNAcentral has a corresponding external ID, with related taxID and ncRNA type.
# 2. if the corresponding ID exists, and RNAcentral id bucket is empty, add the id/s to the bucket. Nice!
# 3. if the corresponding ID exist, and it is only one, and RNAcentral id bucket is not empty: substitute RNAcentral id bucket with the new RNAcentral ID
# 4. if the corresponding ID exist, and it is > 1, and RNAcentral id bucket is not empty: add new RNAcentral IDs, if not present already.

# Keep track of point 1, 2, 3 and 4 numbers (How many we added.)


qrc.external_to_rnacentral('Gm24029', '10090')
 

,RNAcentral_ID,rna_type,external_db,external_id
0,URS0000639553,snoRNA,MGI,MGI:5453806


In [91]:
import pandas as pd
import numpy as np

STANDARD_RNA_TYPES = {
    "tRNA", "miRNA", "rRNA", "pseudo", "lncRNA", "piRNA",
    "sRNA", "lincRNA", "ncRNA", "ribozyme", "snoRNA",
    "scaRNA", "snRNA", "scRNA", "sncRNA"
}
ACCEPTABLE_TYPE_MAP = {
    "SRP_RNA": ["ncRNA"],
    "RNase_MRP_RNA": ["ncRNA", "ribozyme"],
    "pre_miRNA": ["miRNA"],          # precursor -> mature
    "scaRNA": ["snoRNA", "ncRNA"],
    "snRNA": ["ncRNA"]               # small nuclear RNA -> generic ncRNA
}


stats = {
    "matched_external_id": 0,
    "bucket_filled_empty": 0,
    "bucket_replaced_single": 0,
    "bucket_extended_multiple": 0,
    "bucket_extended_multiple_notranscript": 0
}

errors_standard_types = []
errors_no_match_types = []

def flatten_bucket(bucket):
    flat = []
    for x in bucket:
        if isinstance(x, list):
            flat.extend(x)
        else:
            flat.append(x)
    return flat

for idx, row in rna_entities_trial_stepFour.iterrows():

    ext_id = row["Symbol"]
    taxid = str(row["Species"])
    row_rna_type = row["Category"]

    id_buckets = row["ID_buckets"]
    bucket = flatten_bucket(id_buckets.get("RNAcentral", []))

    df_match = qrc.external_to_rnacentral(ext_id, taxid)

    if df_match.empty:
        continue

    stats["matched_external_id"] += 1

    accepted_rows = []

    for _, r in df_match.iterrows():
        rc_id = r["RNAcentral_ID"]
        rc_type = r["rna_type"]
        rc_db = r.get("external_db")

        if pd.isna(rc_id) or pd.isna(rc_type):
            continue

        if rc_type not in STANDARD_RNA_TYPES:
            errors_standard_types.append(
                f"[UNSTANDARDISED RNA TYPE] {rc_type} for row type {row_rna_type}"
            )

        if rc_type == row_rna_type:
            accepted_rows.append(r)
        elif rc_type in ACCEPTABLE_TYPE_MAP and row_rna_type in ACCEPTABLE_TYPE_MAP[rc_type]:
            accepted_rows.append(r)
        else:
            errors_no_match_types.append(
                f"[no match RNA TYPE] {rc_type} for row type {row_rna_type}"
            )

    if not accepted_rows:
        continue

    df_resulting = pd.DataFrame(accepted_rows)

    # Prefer RefSeq-backed mappings if ambiguity
    if len(df_resulting) > 1 and "external_db" in df_resulting.columns:
        df_refseq = df_resulting[df_resulting["external_db"] == "REFSEQ"]
        if not df_refseq.empty:
            df_resulting = df_refseq

    new_ids = df_resulting["RNAcentral_ID"].unique().tolist()

    # ---- bucket logic ----
    bucket_set = set(bucket)
    new_set = set(new_ids)
    
    if len(bucket_set) == 0:
        # nothing existed → initialize
        bucket = sorted(new_set)
        stats["bucket_filled_empty"] += 1
    
    elif bucket_set.intersection(new_set):
        # at least one ID matches → treat new mapping as a refinement
        # (replace with the coherent new set)
        bucket = sorted(new_set)
        stats["bucket_replaced_single"] += 1
    
    else:
        # completely new evidence → extend, do not overwrite
        bucket = sorted(bucket_set.union(new_set))
    
        if len(new_set) == 1:
            stats["bucket_extended_multiple_notranscript"] += 1
        else:
            stats["bucket_extended_multiple"] += 1
    
    id_buckets["RNAcentral"] = bucket
    rna_entities_trial_stepFour.at[idx, "ID_buckets"] = id_buckets


In [92]:
stats # new

{'matched_external_id': 13925,
 'bucket_filled_empty': 867,
 'bucket_replaced_single': 11037,
 'bucket_extended_multiple': 96,
 'bucket_extended_multiple_notranscript': 1149}

In [93]:
compute_stats(rna_entities_trial_stepFour) # new

=== Row-level (entity-level) statistics ===
Ensembl: rows=3889 | IDs/row: max=1, mean=1.00, median=1.0
RNAcentral: rows=15621 | IDs/row: max=454, mean=2.29, median=1.0
NCBI: rows=21192 | IDs/row: max=1, mean=1.00, median=1.0
miRBase: rows=6723 | IDs/row: max=3, mean=1.00, median=1.0
GenBank: rows=3 | IDs/row: max=1, mean=1.00, median=1.0
circBase: rows=1 | IDs/row: max=1, mean=1.00, median=1.0

=== IDs per row (binned) ===
Ensembl: 1=3889, 2=0, 3–10=0, >10=0
RNAcentral: 1=11886, 2=1567, 3–10=1758, >10=410
NCBI: 1=21192, 2=0, 3–10=0, >10=0
miRBase: 1=6716, 2=6, 3–10=1, >10=0
GenBank: 1=3, 2=0, 3–10=0, >10=0
circBase: 1=1, 2=0, 3–10=0, >10=0

=== ID-level (secondary) statistics ===
Ensembl: 3889 unique IDs
RNAcentral: 31467 unique IDs
NCBI: 20784 unique IDs
miRBase: 6584 unique IDs
GenBank: 3 unique IDs
circBase: 1 unique IDs

=== Sequence availability ===
RNAcentral: rows with IDs = 15621
RNAcentral: rows without IDs = 19829
RNAcentral: fraction retained = 0.441
new sequence avibility c

In [94]:
rna_entities_trial_stepFour.to_pickle("rna_entities_trial_stepFour.pkl")

<h2>RNAcentral sequence retrieval, and GO terms retrieval</h2>

In [ ]:
rna_entities_trial_stepFour = pd.read_pickle("rna_entities_trial_stepFour.pkl")

In [ ]:
# input dataframe: rna_entities_trial_stepFour
# 1. retrieve RNAcentral sequences: rnacentral_to_sequence -> list in 'sequences' column
# 2. retrieve GO terms and rfam_identifiers with rnacentral_metadata -> add 'GO' column, NaN if no GO, otherwhise list of correlated GO terms
# 3. RfamId, same as GO terms


In [95]:
from collections import defaultdict
import pandas as pd
import re
from tqdm import tqdm

# ----------------------------
# Helper functions
# ----------------------------

def normalize_rnacentral_bucket(bucket):
    """
    Accepts:
      - []
      - ['URS...', 'URS...']
      - [['URS...', 'URS...'], ['URS...']]
    Returns:
      - flat list[str]
    """
    if not bucket:
        return []

    flat = []
    for x in bucket:
        if isinstance(x, list):
            flat.extend(x)
        else:
            flat.append(x)
    return flat


def clean_rna_sequence(seq: str) -> str:
    """
    Clean RNA sequence:
    - uppercase
    - convert T -> U
    - keep only A/U/C/G
    """
    seq = seq.upper().replace("T", "U")
    seq = re.sub(r"[^AUCG]", "", seq)
    return seq

def get_rnacentral_sequences(id_list):
    """
    Retrieve sequences from RNAcentral for a list of RNAcentral IDs
    Returns a list of sequences (one per ID)
    """
    sequences = []
    for rc_id in id_list:
        try:
            seq = qrc.rnacentral_to_sequence(rc_id)
            # take first sequence if multiple, clean it
            s = seq[0]
            sequences.append(clean_rna_sequence(s))
        except Exception:
            continue
    return sequences

def get_rnacentral_metadata(id_list):
    """
    Retrieve GO terms and Rfam IDs from RNAcentral for a list of RNAcentral IDs
    Returns two lists: go_terms, rfam_ids
    """
    go_terms = set()
    rfam_ids = set()
    for rc_id in id_list:
        try:
            df_meta = qrc.rnacentral_metadata(rc_id)
            if df_meta is not None and not df_meta.empty:
                go_terms.update(df_meta['go_id'].dropna().unique())
                rfam_ids.update(df_meta['rfam_id'].dropna().unique())
        except Exception:
            continue
    return list(go_terms) if go_terms else pd.NA, list(rfam_ids) if rfam_ids else pd.NA

# ----------------------------
# Apply to DataFrame
# ----------------------------

tqdm.pandas(desc="Fetching RNAcentral sequences & metadata")

def _ensure_list(x):
    """Normalize pandas cell content to a list."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if isinstance(x, list):
        return x
    return [x]  # defensive: single stray value


def enrich_rnacentral(row):
    """
    Merge RNAcentral-derived annotations without overwriting existing ones.
    """
    row = row.copy()

    rc_bucket = row.get("ID_buckets", {}).get("RNAcentral", [])
    rc_ids = normalize_rnacentral_bucket(rc_bucket)

    # ---- sequences ----
    existing_seqs = _ensure_list(row.get("sequences"))
    new_seqs = _ensure_list(get_rnacentral_sequences(rc_ids))

    merged_seqs = list(dict.fromkeys(existing_seqs + new_seqs))
    row["sequences"] = merged_seqs

    # ---- metadata ----
    go_list, rfam_list = get_rnacentral_metadata(rc_ids)

    existing_go = _ensure_list(row.get("GO"))
    existing_rfam = _ensure_list(row.get("Rfam"))

    row["GO"] = list(dict.fromkeys(existing_go + _ensure_list(go_list)))
    row["Rfam"] = list(dict.fromkeys(existing_rfam + _ensure_list(rfam_list)))

    return row


rna_entities_final = rna_entities_trial_stepFour.progress_apply(
    enrich_rnacentral,
    axis=1
)



Fetching RNAcentral sequences & metadata: 100%|███████████████████████████████████████████████████████████████████████████████████████████| 35450/35450 [05:43<00:00, 103.08it/s]


In [96]:
import pandas as pd

# Count rows with at least one sequence
count_with_seq = rna_entities_final['sequences'].apply(lambda x: bool(x)).sum()

def has_valid_entry(x):
    if not isinstance(x, list):
        return False
    # keep only non-missing values
    return any(pd.notna(v) for v in x)

# Count rows with at least one real GO term
count_with_go = rna_entities_final['GO'].apply(has_valid_entry).sum()

# Count rows with at least one real Rfam ID
count_with_rfam = rna_entities_final['Rfam'].apply(has_valid_entry).sum()


# Count rows with at least one GO term
#count_with_go = rna_entities_final['GO'].apply(lambda x: isinstance(x, list) and len(x) > 0).sum() # [<NA>]?? conts as one

# Count rows with at least one Rfam ID
#count_with_rfam = rna_entities_final['Rfam'].apply(lambda x: isinstance(x, list) and len(x) > 0).sum()

total_rows = len(rna_entities_final)

print(f"Total rows: {total_rows}")
print(f"Rows with sequences: {count_with_seq} ({count_with_seq/total_rows:.2%})")
print(f"Rows with GO terms: {count_with_go} ({count_with_go/total_rows:.2%})")
print(f"Rows with Rfam IDs: {count_with_rfam} ({count_with_rfam/total_rows:.2%})")


# ---- total number of sequences (including duplicates) ----
total_sequences = rna_entities_final['sequences'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
).sum()

# ---- total number of unique sequences ----
all_sequences = set()

for seq_list in rna_entities_final['sequences']:
    if isinstance(seq_list, list):
        all_sequences.update(seq_list)

total_unique_sequences = len(all_sequences)

print(f"Total sequences (with redundancy): {total_sequences}")
print(f"Total unique sequences: {total_unique_sequences}")
print(f"Redundancy ratio: {1 - total_unique_sequences/total_sequences:.2%}" if total_sequences > 0 else "No sequences")


# ---- length stratification ----
short_threshold = 4096

total_short = 0
total_long = 0

for seq_list in rna_entities_final['sequences']:
    if isinstance(seq_list, list):
        for seq in seq_list:
            if len(seq) < short_threshold:
                total_short += 1
            else:
                total_long += 1

total_sequences_check = total_short + total_long

print(f"Sequences < {short_threshold} nt: {total_short} ({total_short/total_sequences_check:.2%})")
print(f"Sequences ≥ {short_threshold} nt: {total_long} ({total_long/total_sequences_check:.2%})")


# ---- length stratification ----
short_threshold = 1024

total_short = 0
total_long = 0

for seq_list in rna_entities_final['sequences']:
    if isinstance(seq_list, list):
        for seq in seq_list:
            if len(seq) < short_threshold:
                total_short += 1
            else:
                total_long += 1

total_sequences_check = total_short + total_long

print(f"Sequences < {short_threshold} nt: {total_short} ({total_short/total_sequences_check:.2%})")
print(f"Sequences ≥ {short_threshold} nt: {total_long} ({total_long/total_sequences_check:.2%})")


Total rows: 35450
Rows with sequences: 15694 (44.27%)
Rows with GO terms: 1906 (5.38%)
Rows with Rfam IDs: 2170 (6.12%)
Total sequences (with redundancy): 37049
Total unique sequences: 32694
Redundancy ratio: 11.75%
Sequences < 4096 nt: 35678 (96.30%)
Sequences ≥ 4096 nt: 1371 (3.70%)
Sequences < 1024 nt: 24764 (66.84%)
Sequences ≥ 1024 nt: 12285 (33.16%)


In [97]:
rna_entities_final.to_pickle("rna_entities_final.pkl")

<h2>FInalization: Joining Sequence Information with Interaction datset</h2>
<h6>We keep only interaction with avaiable sequences in both Interacting ncRNAs</h6>

In [ ]:
## import pandas as pd

# load interactions
df_filt = pd.read_pickle("interactions_processed_noRNAcentralIDs.pkl")
rna_entities_final = pd.read_pickle("rna_entities_final.pkl")
# ----------------------------
# Helper: canonical entity key
# ----------------------------
def make_entity_key(symbol, category, species, raw_id_list):
    """
    Key: (symbol:str, category:str, species:str, sorted_tuple_of_ids)
    Works if raw_id_list is list, tuple, or other iterable; returns () if empty.
    """
    # normalize inputs to strings
    symbol_s = "" if pd.isna(symbol) else str(symbol)
    category_s = "" if pd.isna(category) else str(category)
    species_s = "" if pd.isna(species) else str(species)

    # normalize raw_id_list to iterable of strings
    if raw_id_list is None:
        ids = []
    elif isinstance(raw_id_list, (list, tuple, set)):
        ids = [str(x) for x in raw_id_list if x is not None]
    else:
        # fallback: single scalar
        ids = [str(raw_id_list)]

    id_tuple = tuple(sorted(ids))
    return (symbol_s, category_s, species_s, id_tuple)

# ----------------------------
# Ensure rna_entities_final exists and keep only entities with sequences
# ----------------------------
# rna_entities_final must be present in namespace
try:
    _ = rna_entities_final
except NameError:
    raise RuntimeError("rna_entities_final not found. Run enrichment step first.")

# keep only rows that have at least one sequence (list with length>0)
rna_entities_final = rna_entities_final.copy()
rna_entities_final = rna_entities_final[
    rna_entities_final['sequences'].apply(lambda x: isinstance(x, list) and len(x) > 0)
].reset_index(drop=True)
print("RNA entities with sequences:", len(rna_entities_final))

# ----------------------------
# Add canonical entity_key to rna_entities_final
# ----------------------------
# Accept different column names for raw id key: prefer 'Raw_ID_key' then 'Raw_IDs' then 'Raw_IDs'
raw_id_col_candidates = ['Raw_ID_key', 'Raw_IDs', 'Raw_IDs_list', 'Raw_IDs']  # fallback list
def pick_raw_id_col(df):
    for c in raw_id_col_candidates:
        if c in df.columns:
            return c
    return None

raw_col = pick_raw_id_col(rna_entities_final)
if raw_col is None:
    # if none, try to fall back to 'ID_buckets' -> cannot derive Raw_ID_key; require user to supply
    raise RuntimeError("No raw ID column found in rna_entities_final (expected Raw_ID_key or similar).")

rna_entities_final['entity_key'] = rna_entities_final.apply(
    lambda r: make_entity_key(
        r.get('Symbol'),
        r.get('Category'),
        r.get('Species'),
        r.get(raw_col)
    ),
    axis=1
)

# ----------------------------
# Build rna_map dataframe for merges
# ----------------------------
rna_map = rna_entities_final[[
    'entity_key', 'sequences', 'GO', 'Rfam', 'ID_buckets'
]].copy()

# ----------------------------
# Build composite keys for interactions (both interactors)
# ----------------------------
df_filt_merged = df_filt.copy()

# ensure Raw_ID lists exist in df_filt_merged with expected names
# expected: 'Raw_ID1_list' and 'Raw_ID2_list' based on earlier pipeline
if 'Raw_ID1_list' not in df_filt_merged.columns or 'Raw_ID2_list' not in df_filt_merged.columns:
    raise RuntimeError("df_filt missing Raw_ID1_list or Raw_ID2_list columns.")

df_filt_merged['key1'] = df_filt_merged.apply(
    lambda r: make_entity_key(
        r.get('Interactor1.Symbol'),
        r.get('Category1'),
        r.get('Species1_TaxID'),
        r.get('Raw_ID1_list')
    ),
    axis=1
)

df_filt_merged['key2'] = df_filt_merged.apply(
    lambda r: make_entity_key(
        r.get('Interactor2.Symbol'),
        r.get('Category2'),
        r.get('Species2_TaxID'),
        r.get('Raw_ID2_list')
    ),
    axis=1
)

# ----------------------------
# Merge sequences/metadata for interactor1
# ----------------------------
# right-side columns will be renamed to avoid collision
rna_map_1 = rna_map.rename(columns={
    'sequences': 'sequences_1',
    'GO': 'GO_1',
    'Rfam': 'Rfam_1',
    'ID_buckets': 'ID_buckets_1'
})

df_filt_merged = df_filt_merged.merge(
    rna_map_1,
    left_on='key1',
    right_on='entity_key',
    how='left'
)

# drop extra entity_key column produced by merge
if 'entity_key' in df_filt_merged.columns:
    df_filt_merged.drop(columns=['entity_key'], inplace=True)

# ----------------------------
# Merge sequences/metadata for interactor2
# ----------------------------
rna_map_2 = rna_map.rename(columns={
    'sequences': 'sequences_2',
    'GO': 'GO_2',
    'Rfam': 'Rfam_2',
    'ID_buckets': 'ID_buckets_2'
})

df_filt_merged = df_filt_merged.merge(
    rna_map_2,
    left_on='key2',
    right_on='entity_key',
    how='left'
)

if 'entity_key' in df_filt_merged.columns:
    df_filt_merged.drop(columns=['entity_key'], inplace=True)

# ----------------------------
# Normalize merged column names to final names
# ----------------------------
# create consistent column names used downstream
df_filt_merged = df_filt_merged.rename(columns={
    'sequences_1': 'seq1_list',
    'GO_1': 'GO1',
    'Rfam_1': 'Rfam1',
    'ID_buckets_1': 'ID_buckets1',
    'sequences_2': 'seq2_list',
    'GO_2': 'GO2',
    'Rfam_2': 'Rfam2',
    'ID_buckets_2': 'ID_buckets2'
})

# If any missing sequence columns were not created by merge, ensure they exist
for c in ['seq1_list', 'seq2_list', 'GO1', 'GO2', 'Rfam1', 'Rfam2', 'ID_buckets1', 'ID_buckets2']:
    if c not in df_filt_merged.columns:
        df_filt_merged[c] = pd.NA

# ----------------------------
# Step: drop interactions without sequences on either side
# ----------------------------
has_seq1 = df_filt_merged['seq1_list'].apply(lambda x: isinstance(x, list) and len(x) > 0)
has_seq2 = df_filt_merged['seq2_list'].apply(lambda x: isinstance(x, list) and len(x) > 0)

df_filt_final = df_filt_merged[has_seq1 & has_seq2].reset_index(drop=True)

print(f"Total interactions before filtering: {len(df_filt)}")
print(f"Total interactions after filtering: {len(df_filt_final)}")


In [99]:
df_filt_final.to_pickle("interactions_with_sequences_FINAL_NEW.pkl")

In [100]:
import pandas as pd

# load interactions
#df_filt_final = pd.read_pickle("FINAL_DATASET_interactions_ncRNAs.pkl")

# Collect all entity_keys present in the filtered interactions
keys_retained = pd.concat([
    df_filt_final['key1'],
    df_filt_final['key2']
]).drop_duplicates()

print(f"Unique RNA entities retained: {len(keys_retained)}")


Unique RNA entities retained: 11929


In [101]:
def count_interactions_under_length(df, max_len):

    kept_rows = []

    for _, row in df.iterrows():

        seq1 = row['seq1_list']
        seq2 = row['seq2_list']

        if not isinstance(seq1, list) or not isinstance(seq2, list):
            continue

        valid_1 = any(len(s) <= max_len for s in seq1)
        valid_2 = any(len(s) <= max_len for s in seq2)

        if valid_1 and valid_2:
            kept_rows.append(row)

    df_kept = pd.DataFrame(kept_rows)

    total_rows = len(df_kept)

    print(f"\nMax length threshold: {max_len} nt")
    print(f"Total interactions: {total_rows}")

    # ----------------------------
    # Row-level coverage
    # ----------------------------
    count_with_seq = (
        df_kept['seq1_list'].apply(lambda x: isinstance(x, list) and len(x) > 0) &
        df_kept['seq2_list'].apply(lambda x: isinstance(x, list) and len(x) > 0)
    ).sum()

    def has_valid_list(x):
        return isinstance(x, list) and any(pd.notna(v) and v != "" for v in x)
    
    # Masks per side
    go1 = df_kept['GO1'].apply(has_valid_list)
    go2 = df_kept['GO2'].apply(has_valid_list)
    
    rfam1 = df_kept['Rfam1'].apply(has_valid_list)
    rfam2 = df_kept['Rfam2'].apply(has_valid_list)
    
    # BOTH sides annotated
    count_go_both = (go1 & go2).sum()
    count_rfam_both = (rfam1 & rfam2).sum()
    
    # AT LEAST ONE side annotated
    count_with_go = (go1 | go2).sum()
    count_with_rfam = (rfam1 | rfam2).sum()

    print(f"Interactions with sequences (both sides): {count_with_seq} ({count_with_seq/total_rows:.2%})")
    print(f"Interactions with GO terms (both sides): {count_go_both} ({count_go_both/total_rows:.2%})")
    print(f"Interactions with GO terms (one side at least): {count_with_go} ({count_with_go/total_rows:.2%})")
    print(f"Interactions with Rfam IDs (both sides): {count_rfam_both} ({count_rfam_both/total_rows:.2%})")
    print(f"Interactions with Rfam IDs (one side at least): {count_with_rfam} ({count_with_rfam/total_rows:.2%})")

    # ----------------------------
    # Sequence-level statistics
    # ----------------------------

    total_sequences = 0
    all_sequences = set()

    for col in ['seq1_list', 'seq2_list']:
        for seq_list in df_kept[col]:
            if isinstance(seq_list, list):
                total_sequences += len(seq_list)
                all_sequences.update(seq_list)

    total_unique_sequences = len(all_sequences)

    print(f"Total sequences (with redundancy): {total_sequences}")
    print(f"Total unique sequences: {total_unique_sequences}")

    if total_sequences > 0:
        redundancy = 1 - total_unique_sequences / total_sequences
        print(f"Redundancy ratio: {redundancy:.2%}")
    else:
        print("Redundancy ratio: NA")

    keys_retained = pd.concat([
        df_kept['key1'],
        df_kept['key2']
    ]).drop_duplicates()
    
    print(f"Unique RNA entities retained: {len(keys_retained)}")

    return df_kept



In [102]:
df_1000000 = count_interactions_under_length(df_filt_final, 1000000)


Max length threshold: 1000000 nt
Total interactions: 82171
Interactions with sequences (both sides): 82171 (100.00%)
Interactions with GO terms (both sides): 6314 (7.68%)
Interactions with GO terms (one side at least): 18164 (22.11%)
Interactions with Rfam IDs (both sides): 6584 (8.01%)
Interactions with Rfam IDs (one side at least): 21230 (25.84%)
Total sequences (with redundancy): 439193
Total unique sequences: 26994
Redundancy ratio: 93.85%
Unique RNA entities retained: 11929


In [104]:
df_10000 = count_interactions_under_length(df_filt_final, 10000)
df_4096 = count_interactions_under_length(df_filt_final, 4096)
df_2048 = count_interactions_under_length(df_filt_final, 2048)
df_1024 = count_interactions_under_length(df_filt_final, 1024)


Max length threshold: 10000 nt
Total interactions: 81458
Interactions with sequences (both sides): 81458 (100.00%)
Interactions with GO terms (both sides): 6293 (7.73%)
Interactions with GO terms (one side at least): 17866 (21.93%)
Interactions with Rfam IDs (both sides): 6537 (8.02%)
Interactions with Rfam IDs (one side at least): 20671 (25.38%)
Total sequences (with redundancy): 436769
Total unique sequences: 26819
Redundancy ratio: 93.86%
Unique RNA entities retained: 11849

Max length threshold: 4096 nt
Total interactions: 75458
Interactions with sequences (both sides): 75458 (100.00%)
Interactions with GO terms (both sides): 6113 (8.10%)
Interactions with GO terms (one side at least): 15832 (20.98%)
Interactions with Rfam IDs (both sides): 6349 (8.41%)
Interactions with Rfam IDs (one side at least): 18417 (24.41%)
Total sequences (with redundancy): 409189
Total unique sequences: 25054
Redundancy ratio: 93.88%
Unique RNA entities retained: 10927

Max length threshold: 2048 nt
Tota